In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

# SET PATH
BASE = Path.home() / "Dropbox" / "PhD" / "Courses" / "Econometrics I" / "PS2"
DATA = BASE / "wage.xlsx"

df = pd.read_excel(DATA)
assert (df["wage"] > 0).all()
df["lw"] = np.log(df["wage"])

# group 1 = females, group 2 = males
y1 = df.loc[df["male"] == 0, "lw"].to_numpy()
y2 = df.loc[df["male"] == 1, "lw"].to_numpy()
n1, n2 = len(y1), len(y2)

s1, s2 = y1.var(ddof=1), y2.var(ddof=1)

# T statistic
T = np.sqrt(n1 * n2 / (n1 + n2)) * (np.log(s1) - np.log(s2))

# pooled kurtosis estimate, each group demeaned by its own mean
d1, d2 = y1 - y1.mean(), y2 - y2.mean()
kappa_hat = (n1 + n2) * (np.sum(d1**4) + np.sum(d2**4)) / (np.sum(d1**2) + np.sum(d2**2))**2

# robust test: T / sqrt(kappa_hat - 1) vs N(0,1)
z_rob = T / np.sqrt(kappa_hat - 1)
z_crit = stats.norm.ppf(0.95)
p_rob = stats.norm.sf(z_rob)

# normal-theory F test
F = s1 / s2
F_crit = stats.f.ppf(0.95, n1 - 1, n2 - 1)
p_F = stats.f.sf(F, n1 - 1, n2 - 1)

print(f"n_female = {n1}, n_male = {n2}")
print(f"s1^2 (female) = {s1:.4f}, s2^2 (male) = {s2:.4f}")
print(f"T = {T:.4f}, kappa_hat = {kappa_hat:.4f}")
print()
print("Robust test")
print(f"  T/sqrt(kappa_hat-1) = {z_rob:.4f}, crit = {z_crit:.4f}, p = {p_rob:.4f}, reject = {z_rob > z_crit}")
print("Normal-theory F test")
print(f"  s1^2/s2^2 = {F:.4f}, crit = {F_crit:.4f}, p = {p_F:.4f}, reject = {F > F_crit}")
print()
print(f"Asymptotic check: T/sqrt(2) = {T/np.sqrt(2):.4f} (F test treats T as N(0,2))")

n_female = 1569, n_male = 1727
s1^2 (female) = 0.3979, s2^2 (male) = 0.3664
T = 2.3621, kappa_hat = 7.3781

Robust test
  T/sqrt(kappa_hat-1) = 0.9353, crit = 1.6449, p = 0.1748, reject = False
Normal-theory F test
  s1^2/s2^2 = 1.0859, crit = 1.0845, p = 0.0473, reject = True

Asymptotic check: T/sqrt(2) = 1.6703 (F test treats T as N(0,2))
